[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week6_feature_store_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 6 -- Feature Engineering at Scale (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** Feast feature store, feature views, point-in-time correct retrieval, training-serving skew

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Explain training-serving skew with a concrete CreditDefault-NG example involving PAY_0
- Set up a Feast feature store with credit_features and payment_history_features views
- Generate a training dataset using point-in-time correct retrieval
- Materialise features to the online store and verify sub-20ms serving latency
- Verify training and serving features match for a sample of customers



---

## MONDAY -- "Training-Serving Skew"


In the credit model: PAY_0 is computed as of January 31st in the training batch job. At serving time, PAY_0 is computed from the live bureau as of the prediction request time. For a customer who paid on time in January (PAY_0=0) but missed an October payment (PAY_0=2), the training value is 0 and the serving value is 2. The model predicts using 0 -- the wrong value.

Pause and Predict -- for this customer, does the skew cause the model to overestimate or underestimate default probability? A PAY_0 of 0 (on time) vs 2 (two-month delay) -- which direction does it push the prediction?


### Exercise 6.1 -- Skew identification

Find two other CreditDefault-NG features that could produce training-serving skew. For each: describe how training and serving computations could differ, and which direction the skew pushes predictions.


In [ ]:
# Exercise 6.1: Skew identification
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: Training-Serving Skew --
# Training pipeline (monthly batch):
def get_pay0_for_training(customer_id, snapshot_date='2026-01-31'):
    return db.execute(
        "SELECT PAY_0 FROM credit_accounts "
        "WHERE ID = %s AND statement_date = %s",
        [customer_id, snapshot_date]).scalar()
# Returns January PAY_0 for ALL customers regardless of current state

# Serving pipeline (real-time API):
def get_pay0_for_serving(customer_id):
    return db.execute(
        "SELECT PAY_0 FROM credit_accounts "
        "WHERE ID = %s "
        "ORDER BY statement_date DESC LIMIT 1",       # most recent row, not a fixed date
        [customer_id]).scalar()
# Returns MOST RECENT PAY_0 -- which may differ from January

# For customer 7823: PAY_0=0 in January, PAY_0=2 in October
print('Skew: model sees PAY_0=0 at serving time when true value is PAY_0=2')


### Expected Output (exact — pure `print()`, no external dependency)

```
Skew: model sees PAY_0=0 at serving time when true value is PAY_0=2
```
The two SQL queries above look almost identical — same table, same column — but one is
pinned to a fixed historical date and the other always returns the latest row. That one-word
difference (`AND statement_date = %s` vs `ORDER BY statement_date DESC LIMIT 1`) is the entire
skew bug. This is why skew is so easy to ship accidentally: nothing here throws an error.


### Monday Morning Moment

*Slack -- Monday, 10:30am.*

**Temi Adeyemi:** Is training-serving skew the same as data drift?

**Dr. Emeka Obi:** Different problems, different fixes. Drift: the input distribution changed because the world changed. Skew: the feature computation produces different values at training vs serving time for the same customer. Drift is external. Skew is internal.

**Temi Adeyemi:** How do I tell which is happening?

**Dr. Emeka Obi:** Skew: compare training features to serving features for the same customer at the same time. If they differ, you have skew. They can happen simultaneously -- as they are here.

**Sola Fashola:** The Feast setup fixes the skew. The retraining pipeline from Week 5 fixes the drift. Two separate problems, two separate fixes.




---

## TUESDAY -- "Feast Feature Store Setup"


Feast has two components: the offline store (historical features for training, backed by parquet files) and the online store (low-latency lookup for serving, backed by Redis). Feature definitions: Entity (the primary key), FeatureView (a set of features from one source with a TTL), and FeatureService (a named collection of features for a specific use case).


### Exercise 6.2 -- Feature store registration

Register credit_features and payment_history_features in Feast. payment_history_features should include PAY_2, PAY_3, PAY_4, PAY_AMT2, PAY_AMT3. Verify with feast features list.


In [ ]:
# Exercise 6.2: Feature store registration
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: Feast Feature Store Setup --
from feast import Entity, FeatureStore, FeatureView, Feature, FileSource, ValueType
from feast.types import Float64, Int64
from datetime import timedelta

customer = Entity(name='customer_id', value_type=ValueType.INT64,
                  description='CreditDefault-NG customer account')

credit_source = FileSource(
    path='data/credit_features.parquet',
    timestamp_field='event_timestamp')

credit_fv = FeatureView(
    name='credit_features',
    entities=[customer],
    ttl=timedelta(days=35),  # one monthly billing cycle
    source=credit_source,
    schema=[
        Feature(name='LIMIT_BAL', dtype=Float64),
        Feature(name='PAY_0',     dtype=Int64),
        Feature(name='PAY_2',     dtype=Int64),
        Feature(name='BILL_AMT1', dtype=Float64),
        Feature(name='PAY_AMT1',  dtype=Float64),
    ])

fs = FeatureStore(repo_path='feast/')
fs.apply([customer, credit_fv])
print('Feature store registered')


### Expected Output (illustrative)

```
Feature store registered
```
Behind that one line, Feast has now written entity and feature-view definitions to
`feast/registry.db` — run `feast feature-views list` afterward and you should see
`credit_features` with its five registered features and 35-day TTL.



---

## WEDNESDAY -- "Point-in-Time Correct Training Data"


Point-in-time correct retrieval returns feature values as they existed at a specified timestamp. When you provide (customer_id, event_timestamp) pairs, Feast returns each customer's feature values as of their specific timestamp -- not as of today. This prevents leakage: a customer's October payment status cannot leak into January training data.

Pause and Predict -- for customer 7823 with a payment event on 2026-09-15 changing PAY_0 from 0 to 2: what does Feast return for event_timestamp=2026-08-31? For event_timestamp=2026-10-01?


### Exercise 6.3 -- Point-in-time test

For customer_id=7823 with a payment event on 2026-09-15: verify get_historical_features with event_timestamp=2026-08-31 returns PAY_0=0, and event_timestamp=2026-10-01 returns PAY_0=2. Confirms point-in-time correctness.


In [ ]:
# Exercise 6.3: Point-in-time test
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: Point-in-Time Correct Training Data --
import pandas as pd
from feast import FeatureStore

fs = FeatureStore(repo_path='feast/')

# Each row is a training example with a specific label timestamp
entity_df = pd.DataFrame({
    'customer_id':     [7823, 7824, 7825],
    'event_timestamp': pd.to_datetime(['2026-01-31', '2026-04-30', '2026-10-31']),
})

# Feast returns PAY_0 as of each customer's event_timestamp
# Customer 7823: sees January PAY_0 (before the October missed payment)
# Customer 7825: sees October PAY_0 (after the missed payment)
training_df = fs.get_historical_features(
    entity_df=entity_df,
    features=['credit_features:LIMIT_BAL', 'credit_features:PAY_0']
).to_df()
print(training_df)


### Expected Output (illustrative)

```
   customer_id event_timestamp  LIMIT_BAL  PAY_0
0         7823      2026-01-31   150000.0      0
1         7824      2026-04-30   220000.0     -1
2         7825      2026-10-31    80000.0      2
```
Customer 7823's row shows `PAY_0=0` because their `event_timestamp` is January — Feast is
correctly returning what was *true as of that date*, not today's value. This is the property
that "training-serving skew" violates when done wrong: point-in-time correctness during
training, but latest-value lookups at serving time.



---

## THURSDAY -- "Materialise for Online Serving"


Materialise copies the latest feature values from the offline store to Redis (the online store). The serving API then calls fs.get_online_features(), which returns results in under 10ms instead of seconds.


### STOP AND TRACE

Before running:

online = fs.get_online_features([...], [{'customer_id': 7823}]).to_dict()

What timestamp does the online store use for the returned feature values?
What timestamp does get_historical_features use?
When would these return different PAY_0 values for the same customer?
Why this matters: online store serves the latest materialised values. For training you want historical values as of a specific date. If you use the online store for training, future payment events leak into past training examples.


In [ ]:
# Exercise 6.4: STOP AND TRACE -- offline vs online store
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Materialise for Online Serving --
from feast import FeatureStore
from datetime import datetime

fs = FeatureStore(repo_path='feast/')

# Copy latest features to Redis (run after each monthly batch)
fs.materialize(start_date=datetime(2026, 1, 1), end_date=datetime.now())
print('Features materialised to online store')

# Serving: < 10ms feature lookup
def get_serving_features(customer_id: int) -> dict:
    fv = fs.get_online_features(
        features=['credit_features:LIMIT_BAL', 'credit_features:PAY_0',
                  'credit_features:BILL_AMT1', 'credit_features:PAY_AMT1'],
        entity_rows=[{'customer_id': customer_id}]
    ).to_dict()
    return {k: v[0] for k, v in fv.items()}


### Expected Output (illustrative)

```
Features materialised to online store
```
After this call, `get_serving_features(7823)` should return the *current* PAY_0 (2, if that's
the latest statement), which is correct for serving — the online store is deliberately
"latest value," while the offline store used in Wednesday's exercise is deliberately
"point-in-time." Both are correct; they answer different questions.



---

## FRIDAY -- "The Friday Build: Skew Verification"


Verify training and serving features match for 50 random Q1 customers. Compare offline features (as of January 31st) with online store features (latest materialised). PAY_0 mismatches are expected for customers who made payments since January -- those are not skew, they are legitimate changes. What you are verifying is that the computation logic is consistent.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Skew Verification --
sample_ids = pd.read_csv('datasets/creditdefault_q1.csv').ID.sample(50, random_state=42)
mismatch = 0
for cid in sample_ids:
    train_f   = get_training_features(cid, as_of='2026-01-31')
    serving_f = get_serving_features(cid)
    if abs(train_f.get('LIMIT_BAL', 0) - serving_f.get('credit_features:LIMIT_BAL', 0)) > 1:
        print(f'SKEW: customer {cid} LIMIT_BAL mismatch')
        mismatch += 1
print(f'Skew check: {mismatch} mismatches')
print('PAY_0 mismatches are expected for customers with payments since January -- not skew')


### Expected Output (illustrative)

```
Skew check: 0 mismatches
PAY_0 mismatches are expected for customers with payments since January -- not skew
```
Zero `LIMIT_BAL` mismatches is the pass condition — `LIMIT_BAL` rarely changes between
January and now, so any mismatch there is a real skew bug. `PAY_0` mismatches, by contrast,
are expected and correct, since that value legitimately changes month to month.


### Friday Workplace Moment

*DataFlow -- Friday standup.*

**Sola Fashola:** Skew check. What did you find?

**Temi Adeyemi:** 47 of 50 customers matched exactly on LIMIT_BAL. 3 had PAY_0 differences. In all 3 cases, the customers made payments between January and the materialisation date -- legitimate changes, not pipeline errors.

**Dr. Emeka Obi:** So the skew is eliminated. The remaining differences are expected feature evolution.

**Temi Adeyemi:** Yes. Feast is working correctly. The computation logic is now consistent between training and serving.

**Dr. Emeka Obi:** Document that distinction in the feature store README. The next engineer needs to know the difference.



## YOUR WEEK 6 CHECKLIST

Check each item before moving on.

- [ ] I can explain training-serving skew with a specific CreditDefault-NG example.
- [ ] Both feature views are registered in Feast and appear in feast features list.
- [ ] Point-in-time correctness verified: past timestamp returns the historically correct value.
- [ ] Online store serves features with confirmed latency under 20ms.
- [ ] Skew verification distinguishes pipeline errors from expected feature evolution.


## Challenge Task

> **Core path:** Implement a feature freshness monitor: alert if any feature is more than 35 days stale (one billing cycle).

> **Advanced path:** Build a backfill pipeline for payment_history_features for every month from January to October. Verify retrieval works for any month in the backfill window.


## Common Mistakes

**Treating training-serving skew as data drift:** Skew requires fixing the feature pipeline. Drift requires retraining. A new model trained on skewed data has the same skew.

**Materialising too infrequently:** If materialisation is monthly but payment events are daily, serving features can be 30 days stale. Match frequency to the rate of change of your most time-sensitive features.

**Ignoring the TTL parameter:** A FeatureView with ttl=35 days returns nothing for customers whose last statement was more than 35 days ago -- silent missing features at serving time.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)